# Chinee apple: fine-tuned YOLO11-seg instance segmentation

Stage 3. The CVAT + SAM3 polygons are converted to YOLO segmentation labels and
a **YOLO11-seg** model is fine-tuned end to end, then scored both the
Ultralytics way and the repo's shared way so the row is comparable with
[`dinov3-linear-probe`](https://github.com/nutboltu/chinee-apple-cv-benchmark/tree/main/pipelines/dinov3-linear-probe).

Unlike the probe, nothing here is frozen. That is the point: the probe has no
spatial prior, so canopies fragment and its instance mAP sits at 0.147 against
a patch AP of 0.769. A detector is the principled fix for that gap.

**Why `-seg` and not boxes.** Over the 202 annotated polygons the median
instance covers 20% of its 1024px tile, the median box covers 27%, and 64% of
instances are clipped by a tile edge. A box around a sprawling, edge-truncated
canopy says very little, and two such boxes agree even when the masks do not.

**What you need**
1. The annotated tiles and the CVAT `batch_*.zip` exports in one Drive folder.
2. A GPU runtime. Runtime > Change runtime type > T4.

No gated weights, no DINOv3 clone. COCO-pretrained YOLO11 downloads itself.

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "NO GPU. Runtime > Change runtime type > T4, or this will take hours."
!pip -q install ultralytics
import torch, ultralytics
print("torch      ", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)

## 2. Code

Clone the repo so the three scripts and the shared `metrics` package are
importable. The notebook calls the scripts rather than reimplementing them, so
a run here and a run on the command line train the same model with the same
settings. Replace the URL if you work from a fork.

In [ ]:
import os, sys
REPO = "https://github.com/nutboltu/chinee-apple-cv-benchmark.git"
if not os.path.exists("chinee-apple-cv-benchmark"):
    !git clone -q $REPO
SCRIPTS = "chinee-apple-cv-benchmark/pipelines/yolo11-seg"
sys.path.insert(0, SCRIPTS)
sys.path.insert(0, "chinee-apple-cv-benchmark")   # for the shared metrics package
import evaluate as ev
from metrics import render
print("ok")

## 3. Data

The imagery is not in git. Mount Drive first, then point `DATA` at the folder
holding the tile jpgs (in `batch_01/`.. subfolders or flat) **and** the
`batch_*.zip` CVAT exports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/chinee-apple/processed')   # <- edit me
CRIB = DATA / 'tile_markers.csv'        # marker boxes, used to paint the cardboard out

assert DATA.exists(), f"not found: {DATA}"
print("zips :", len(list(DATA.glob('batch_*.zip'))))
print("tiles:", len(list(DATA.rglob('*.jpg'))))
print("crib :", CRIB.exists())

## 4. Annotations to YOLO labels

Each `<polygon points="x1,y1;x2,y2;...">` becomes one line of
`0 x1/W y1/H x2/W y2/H ...`. The conversion is lossless: rasterising the written
labels back and comparing against the CVAT rings gives IoU 1.000000 on all 180
tiles.

Four decisions are baked into the script, and each one is a way the experiment
could otherwise go quietly wrong.

| decision | why |
| --- | --- |
| only the 180 annotated tiles are written | 440 tiles came back empty, which means nobody annotated them, not that they hold no chinee apple. YOLO trains every pixel outside a ring as background, so including them teaches the model to suppress the plants it should find. |
| the split is whole source frames | tiles are cut 8x6 from one frame and the bottom row overlaps the row above by 684px, so neighbouring tiles share literal pixels. Splitting over tiles leaks train into test. |
| cardboard is painted out of the imagery | each board sits beside its target plant, so a detector can learn "board, therefore canopy next to it" and then find nothing on an unmarked flight. A single-ring polygon cannot carry an ignore region, so it has to be done in image space. |
| polygons are written at full precision | CVAT emits whole-pixel coordinates, which sit exactly on pixel edges, so rounding flips whole scanline spans on these filamentary shapes. |

**The dataset is built on the local disk, not on Drive.** Training reads every
tile once per epoch for hundreds of epochs. Through a mounted Drive that is the
slowest thing in the notebook by a wide margin. Results get copied back to
Drive at the end.

In [ ]:
DS = Path('/content/yolo_dataset')     # local disk on purpose, see above
SEED = 0

!python {SCRIPTS}/cvat_to_yolo.py --zips "{DATA}/batch_*.zip" --tiles "{DATA}" \
    --crib "{CRIB}" --out {DS} --val-frac 0.30 --seed {SEED}

### Look at the labels before training on them

An annotation bug costs a whole training run, and it shows up instantly here.
Green outlines are the converted labels drawn back onto the tiles they came
from. Check that they sit on the brown thorny shrubs, and that where a board was
painted out the patch is textured rather than a flat rectangle.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

labs = sorted((DS/'labels'/'train').glob('*.txt'))
labs = [p for p in labs if p.read_text().strip()][:4]

fig, ax = plt.subplots(1, len(labs), figsize=(4.2*len(labs), 4.4))
ax = np.atleast_1d(ax)
for a, lab in zip(ax, labs):
    img = DS/'images'/'train'/f'{lab.stem}.jpg'
    with Image.open(img) as im:
        size = im.size
        a.imshow(im)
    for ring in ev.read_label(lab, *size):
        xy = np.asarray(ring + [ring[0]])
        a.plot(xy[:, 0], xy[:, 1], c='lime', lw=1.4)
    a.set_title(lab.stem[-20:], fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()
print(f"{len(labs)} tiles shown. Green = the YOLO label the model will train on.")

## 5. Train

`yolo11s-seg` at 1024px. The choices worth knowing about:

- **`s`, about 10M parameters.** The binding constraint is 145 training
  instances over 39 frames, not capacity. `n` and `m` are a sweep in section 8,
  not an upgrade path.
- **1024px, the native tile size.** The objects are large enough that 640 keeps
  their geometry, but the class cue is canopy texture on a brown dry-season
  shrub and that is exactly what downsampling destroys. `IMGSZ = 640` is about
  2.5x faster and is the honest speed comparison.
- **`flipud=0.5`, `degrees=180`.** Ultralytics ships both at zero because COCO
  photographs have a ground plane. These are nadir UAV frames with no canonical
  up, so both are free augmentation, and at 130 training tiles that matters more
  than the model size.
- **`close_mosaic=30`.** Mosaic is the main regulariser at this dataset size,
  but it downscales objects that already run off the tile edge, so the last 30
  epochs are trained on real tiles at real scale.

On a T4 at 1024px expect roughly 25 to 40 seconds an epoch. 300 epochs with
`patience=60` usually stops early. Start with `EPOCHS = 50` if you only want to
see the loop work.

In [ ]:
MODEL  = 'yolo11s-seg.pt'
IMGSZ  = 1024
EPOCHS = 300
BATCH  = 4          # 4 fits yolo11s-seg at 1024px on a 16GB T4. Raise it at 640.

RUNS = Path('/content/runs')
!python {SCRIPTS}/train_yolo.py --data {DS} --model {MODEL} --imgsz {IMGSZ} \
    --epochs {EPOCHS} --batch {BATCH} --seed {SEED} --out {RUNS}

In [ ]:
RUN     = RUNS / f"{Path(MODEL).stem}_{IMGSZ}px"
WEIGHTS = RUN / 'weights' / 'best.pt'
assert WEIGHTS.exists(), f"no weights at {WEIGHTS}, did training finish?"
print(WEIGHTS)

# Ultralytics' own training curves and PR plots
from IPython.display import Image as IPyImage, display
for p in ('results.png', 'MaskPR_curve.png', 'confusion_matrix_normalized.png'):
    f = RUN / p
    if f.exists():
        print(p); display(IPyImage(filename=str(f), width=760))

## 6. Score it the repo's way

Ultralytics already printed a mask mAP, but it runs its own NMS and matches at
full mask resolution, while every other pipeline here is scored by `metrics/` on
a **32x32 patch grid**, which is where a ViT-B/14 probe at 448px can be read.
Putting both in one table would read as a comparison and would not be one, so
they are kept apart: the Ultralytics numbers live in `train_summary.json`, the
comparable ones in `metrics.json`.

Downsampling YOLO to a 32x32 grid throws away real boundary precision that it
has and the probe never had, so the mask IoU below is a **floor** on what this
model can do. `--grid 0` scores at full 1024px resolution for the standalone
figure.

Two instance numbers come out, deliberately:

- `mAP_50_95` from connected components, comparable with the probe
- `native_mAP_50_95` from YOLO's own instances, same 101-point AP convention

Expect native to be higher. If it is not, the model is producing one blob per
tile and the detector is not earning its keep.

In [ ]:
EVAL = Path('/content/eval_out')
!python {SCRIPTS}/evaluate.py --weights {WEIGHTS} --data {DS} \
    --grid 32 --imgsz {IMGSZ} --conf 0.001 --out {EVAL}

### The component metric cannot reach 1.0

`evaluate.py` also reports `component_ceiling_mAP_50_95`: what the connected
component metric scores when handed a **perfect** prediction. It is not 1.0,
because components cannot separate two canopies that touch. On the 17 held-out
frames at grid 32 with the probe's tuned `smooth=1, min_size=8` it is **0.605**,
and 57 annotated plants come back as 49 components.

So dinov3-linear-probe's 0.147 is about 24% of what that metric can award, not
15%. Read any component-based instance figure in this repo against its ceiling,
which is why the ceiling is written into `metrics.json` beside the number it
bounds and never travels without it.

In [ ]:
import json
m = json.loads((EVAL/'metrics.json').read_text())
inst = m['instance']
print(f"components      {inst['mAP_50_95']:.3f} mAP   ceiling {inst['component_ceiling_mAP_50_95']:.3f}"
      f"   -> {100*inst['mAP_50_95']/inst['component_ceiling_mAP_50_95']:.0f}% of achievable")
print(f"own instances   {inst['native_mAP_50_95']:.3f} mAP   ceiling 1.000")
print(f"gt / predicted  {inst['native_n_gt_instances']} / {inst['native_n_pred_instances']}")
print()
print(render.render_one(m, html=False))

### Qualitative check

Numbers hide failure modes. Magenta is predicted, green is the annotation.

Two things to look for. **Canopies the model found that nobody annotated**:
those are counted as false positives here but may well be real plants, because
only the cardboard-marked plant was ever annotated. And **one prediction
spanning two annotated plants**, which is the merge failure that the component
metric also suffers from.

In [ ]:
from ultralytics import YOLO
model = YOLO(str(WEIGHTS))

val = sorted((DS/'images'/'val').glob('*.jpg'))
show = [p for p in val if (DS/'labels'/'val'/f'{p.stem}.txt').read_text().strip()][:4]

fig, ax = plt.subplots(1, len(show), figsize=(4.4*len(show), 4.6))
ax = np.atleast_1d(ax)
for a, p in zip(ax, show):
    r = model.predict(str(p), imgsz=IMGSZ, conf=0.25, retina_masks=True, verbose=False)[0]
    with Image.open(p) as im:
        size = im.size
        a.imshow(im)
    if r.masks is not None and len(r.masks):
        md_ = r.masks.data.cpu().numpy()
        for inst_mask in md_:
            a.contour(inst_mask, levels=[0.5], colors='magenta', linewidths=1.6)
    for ring in ev.read_label(DS/'labels'/'val'/f'{p.stem}.txt', *size):
        xy = np.asarray(ring + [ring[0]])
        a.plot(xy[:, 0], xy[:, 1], c='lime', lw=1.3)
    n = 0 if r.masks is None else len(r.masks)
    a.set_title(f'{p.stem[-18:]}  {n} predicted', fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()
print('magenta = predicted at conf 0.25, green = annotation')

## 7. Save the run to Drive

Written locally first and copied in one step, because writing many small files
straight to a mounted Drive is slow. The folder is timestamped and carries the
model and resolution, so runs never overwrite each other and are still
identifiable months later.

In [ ]:
import shutil, time
RUN_ID    = f"{time.strftime('%Y-%m-%d_%H%M')}_{Path(MODEL).stem}_{IMGSZ}px"
DRIVE_OUT = Path('/content/drive/MyDrive/chinee-apple/yolo11seg') / RUN_ID

LOCAL = Path('/content/run_bundle')
shutil.rmtree(LOCAL, ignore_errors=True)
LOCAL.mkdir(parents=True)

shutil.copy2(WEIGHTS, LOCAL/'best.pt')
shutil.copy2(EVAL/'metrics.json', LOCAL/'metrics.json')
shutil.copy2(DS/'split.json', LOCAL/'split.json')          # the exact split, and its seed
for p in ('train_summary.json', 'results.png', 'results.csv',
          'MaskPR_curve.png', 'args.yaml'):
    if (RUN/p).exists():
        shutil.copy2(RUN/p, LOCAL/p)

DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
shutil.rmtree(DRIVE_OUT, ignore_errors=True)
shutil.copytree(LOCAL, DRIVE_OUT)
shutil.make_archive(str(DRIVE_OUT), 'zip', LOCAL)

files = sorted(DRIVE_OUT.iterdir())
print(f"saved {len(files)} files, {sum(f.stat().st_size for f in files)/1e6:.0f} MB")
print(f"  folder  {DRIVE_OUT}")
print(f"  zip     {DRIVE_OUT}.zip")
print()
print("In Drive, look under  MyDrive / chinee-apple / yolo11seg /", RUN_ID)

## 8. Sweep model size and resolution

The claim that `s` at 1024px is the right default is only worth as much as the
rows beside it. Each row is a full training run, so this is the expensive cell:
five runs at roughly 20 to 40 minutes each on a T4.

Every run reuses the same dataset and therefore the same frame-grouped split, so
the rows are directly comparable. Trim `GRID` to what you have runtime for.

In [ ]:
GRID = [('yolo11n-seg.pt', 1024, 8),
        ('yolo11s-seg.pt', 1024, 4),
        ('yolo11m-seg.pt', 1024, 2),
        ('yolo11s-seg.pt',  640, 16)]

SWEEP = Path('/content/drive/MyDrive/chinee-apple/yolo11seg_sweep') / time.strftime('%Y-%m-%d_%H%M')
SWEEP.mkdir(parents=True, exist_ok=True)
rows = []
for mdl, sz, bs in GRID:
    tag = f"{Path(mdl).stem}_{sz}px"
    print(f"\n===== {tag} =====")
    !python {SCRIPTS}/train_yolo.py --data {DS} --model {mdl} --imgsz {sz} \
        --epochs {EPOCHS} --batch {bs} --seed {SEED} --out {RUNS}
    w = RUNS/tag/'weights'/'best.pt'
    if not w.exists():
        print(f"skipped {tag}, no weights"); continue
    !python {SCRIPTS}/evaluate.py --weights {w} --data {DS} --grid 32 --imgsz {sz} \
        --conf 0.001 --out /content/eval_{tag}
    d = json.loads(Path(f'/content/eval_{tag}/metrics.json').read_text())
    shutil.copy2(f'/content/eval_{tag}/metrics.json', SWEEP/f'{tag}.json')
    rows.append({'run': tag, 'patch AP': d['patch']['ap'], 'IoU': d['mask']['mean_iou'],
                 'comp mAP': d['instance']['mAP_50_95'],
                 'own mAP': d['instance']['native_mAP_50_95'],
                 'ceiling': d['instance']['component_ceiling_mAP_50_95']})

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(SWEEP/'sweep.csv', index=False)
print('\nsaved to', SWEEP)
df

## What to expect

No measured baseline yet, so there is nothing to paste here. Two predictions
worth writing down in advance, because they are what this experiment is for.

**Instance mAP should clearly beat the probe's 0.147**, read against the 0.605
ceiling. A detector has the spatial prior the probe lacks. If it does not win
here, the 180 tile dataset is the binding constraint and the next move is more
annotation, not more model.

**Patch AP may well lose to the probe's 0.769.** A frozen DINOv2 backbone saw
142M images; this fine-tunes from COCO on 130 tiles. Losing per-patch while
winning per-instance is the expected and informative outcome, not a
contradiction, and it is the argument for the third design: a light
convolutional decoder on frozen DINO features.

**The sparse annotation is the floor under all of it.** Only the plant each
cardboard pointed at was annotated, so unmarked chinee apple in a training tile
is taught as background and punished at evaluation. Reported precision is a
lower bound and the true value is unknown, not merely unmeasured. Nothing in
this notebook fixes that; it needs one exhaustively annotated subset to even
quantify.

With 57 held-out instances, every figure here carries wide error bars. Re-run
with `SEED = 1`, `2`, `3` and report the spread.

## 9. Write up the run

Produces the two things to put on Confluence, because dragging files into a wiki
page is not a workflow.

| File | What to do with it |
| --- | --- |
| `report.html` | Open it, copy everything, paste into the Confluence editor using the source or markup view. Every table arrives filled in. |
| `contact_sheet.png` | Drag into the page. One image, the strongest detections. |

Both land in the run folder on Drive beside the weights and `metrics.json`.

In [ ]:
report = DRIVE_OUT / 'report.html'
report.write_text(render.render_one(m, html=True))
print('wrote', report)

# Side by side with the probe, if its metrics.json is on Drive too.
PROBE_METRICS = Path('/content/drive/MyDrive/chinee-apple/probe_out/metrics.json')
if PROBE_METRICS.exists():
    both = [json.loads(PROBE_METRICS.read_text()), m]
    (DRIVE_OUT/'comparison.html').write_text(render.render_compare(both, html=True))
    print('wrote', DRIVE_OUT/'comparison.html')
    print()
    print(render.render_compare(both, html=False))
else:
    print(f'\nno probe metrics at {PROBE_METRICS}, skipping the comparison table')

In [ ]:
# One contact sheet of the strongest detections, instead of 50 separate files.
from PIL import ImageDraw, ImageFont
from IPython.display import Image as IPyImage, display

def _font(sz):
    for f in ('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf'):
        try: return ImageFont.truetype(f, sz)
        except Exception: pass
    return ImageFont.load_default()

COLS, CELL, CAP = 4, 320, 26
scored = []
for p in val:
    r = model.predict(str(p), imgsz=IMGSZ, conf=0.25, retina_masks=True, verbose=False)[0]
    if r.masks is None or not len(r.masks):
        continue
    conf = float(r.boxes.conf.max())
    scored.append((conf, p, r))
scored.sort(key=lambda t: -t[0])
pick = scored[:8]

if not pick:
    print('nothing detected above conf 0.25, no contact sheet')
else:
    rowsn = (len(pick) + COLS - 1) // COLS
    sheet = Image.new('RGB', (COLS*CELL, rowsn*(CELL+CAP)), 'white')
    draw  = ImageDraw.Draw(sheet); font = _font(14)
    for i, (conf, p, r) in enumerate(pick):
        with Image.open(p) as im:
            base = im.convert('RGB')
        ov = Image.new('RGBA', base.size, (0, 0, 0, 0))
        od = ImageDraw.Draw(ov)
        for inst_mask in r.masks.xy:          # polygon in pixel coords
            if len(inst_mask) >= 3:
                od.polygon([tuple(q) for q in inst_mask], outline=(255, 0, 255, 255),
                           fill=(255, 0, 255, 70))
        base = Image.alpha_composite(base.convert('RGBA'), ov).convert('RGB')
        base.thumbnail((CELL, CELL))
        x, y = (i % COLS)*CELL, (i // COLS)*(CELL+CAP)
        sheet.paste(base, (x, y))
        draw.text((x+4, y+CELL+5), f'{p.stem[-22:]}  {len(r.masks)} found, max {conf:.2f}',
                  fill='black', font=font)
    out = DRIVE_OUT / 'contact_sheet.png'
    sheet.save(out)
    print('wrote', out)
    display(IPyImage(filename=str(out), width=900))

### What to pin, and where

1. Open the pipeline page on Confluence.
2. Paste `report.html` into the **Results** section, replacing what is there.
   Do not retype numbers.
3. Drag `contact_sheet.png` in below it.
4. If `comparison.html` was written, paste it into the section that compares
   pipelines. It carries the probe row and this one side by side.
5. Update **Limitations** by hand. That section is judgement, not output.

State the component ceiling next to any instance mAP you quote. A reader who
sees 0.30 against an unstated ceiling of 0.605 will conclude something false.

## Where everything is saved

| Path on Drive | Contents |
| --- | --- |
| `chinee-apple/yolo11seg/<run>/` | `best.pt`, `metrics.json`, `split.json`, the Ultralytics curves, the report and the contact sheet |
| `chinee-apple/yolo11seg/<run>.zip` | the same run as one file |
| `chinee-apple/yolo11seg_sweep/<ts>/` | one `metrics.json` per sweep row plus `sweep.csv` |

`<run>` is the timestamp, model and resolution, so two runs are never confused.
`split.json` carries the frame lists and the seed, which is what makes a run
repeatable rather than merely recorded.

The `yolo_dataset/` folder stays on the Colab disk and dies with the runtime.
That is deliberate: it is derived data, rebuilt in a few seconds by section 4
from the exports and the seed.